Networking RAG - LangGraph Pipeline

Workflow:

START
  ↓
Embed Question
  ↓
Retrieve Chunks from ChromaDB
  ↓
Generate Answer using Groq
  ↓
END

In [1]:
try:

    !pip install -q \
    langgraph \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:

    print("Installation Error:")
    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
try:

    from typing import TypedDict

    from langgraph.graph import (
        StateGraph,
        START,
        END
    )

    print("LangGraph imports successful!")

except Exception as e:

    print("Import Error:")
    print(e)

LangGraph imports successful!


In [3]:
try:

    class GraphState(TypedDict):

        question: str

        query_embedding: list

        retrieved_chunks: list

        answer: str

    print("GraphState created successfully!")

except Exception as e:

    print("State Error:")
    print(e)

GraphState created successfully!


In [4]:
try:

    from google import genai
    from google.colab import userdata

    gemini_client = genai.Client(
        api_key=userdata.get("GEMINI_API_KEY")
    )

    print("Gemini connected successfully!")

except Exception as e:

    print("Gemini Connection Error:")
    print(e)

Gemini connected successfully!


In [5]:
try:

    import pickle
    import chromadb

    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    if collection.count() == 0:

        collection.add(
            ids=[f"chunk_{i}" for i in range(len(chunks))],
            documents=chunks,
            embeddings=chunk_embeddings
        )

    print("ChromaDB connected successfully!")
    print("Records:", collection.count())

except Exception as e:

    print("ChromaDB Error:")
    print(e)

ChromaDB connected successfully!
Records: 182


In [6]:
try:

    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("GROQ_API_KEY")
    )

    print("Groq connected successfully!")

except Exception as e:

    print("Groq Connection Error:")
    print(e)

Groq connected successfully!


In [7]:
try:

    def embed_question(state):

        question = state["question"]

        response = gemini_client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        state["query_embedding"] = (
            response.embeddings[0].values
        )

        print("Question embedding generated!")

        return state

    print("Embed Question Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Embed Question Node created!


In [8]:
try:

    test_state = {
        "question": "What is DNS?"
    }

    result = embed_question(test_state)

    print("Embedding Dimension:")
    print(len(result["query_embedding"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Embedding Dimension:
3072


In [9]:
try:

    def retrieve_chunks(state):

        results = collection.query(
            query_embeddings=[
                state["query_embedding"]
            ],
            n_results=5
        )

        state["retrieved_chunks"] = (
            results["documents"][0]
        )

        print("Top 5 chunks retrieved!")

        return state

    print("Retrieve Chunks Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Retrieve Chunks Node created!


In [10]:
try:

    state = {
        "question": "What is DNS?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    print("\nRetrieved Chunks:")
    print(len(state["retrieved_chunks"]))

except Exception as e:

    print("Test Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!

Retrieved Chunks:
5


In [11]:
try:

    for i, chunk in enumerate(
        state["retrieved_chunks"],
        start=1
    ):

        print("=" * 60)
        print(f"Chunk {i}")
        print(chunk[:500])
        print()

except Exception as e:

    print(e)

Chunk 1
The Domain Name System (DNS) is a hierarchical and distributed name service that provides a naming system for computers, services, and other resources on the Internet or other Internet Protocol (IP) networks. It associates various information with domain names (identification strings) assigned to each of the associated entities. Most prominently, it translates readily memorized domain names to the numerical IP addresses needed for locating and identifying computer services and devices with the u

Chunk 2
Function
An often-used analogy to explain the DNS is that it serves as the phone book for the Internet by translating human-friendly computer hostnames into IP addresses. For example, the hostname www.example.com within the domain name example.com translates to the addresses 93.184.216.34 (IPv4) and 2606:2800:220:1:248:1893:25c8:1946 (IPv6). The DNS can be quickly and transparently updated, allowing a service's location on the network to change without affecting the end users, 

In [12]:
try:

    def generate_answer(state):

        context = "\n\n".join(
            state["retrieved_chunks"]
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not available in the context,
respond with:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{state["question"]}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        state["answer"] = (
            response.choices[0].message.content
        )

        print("Answer generated successfully!")

        return state

    print("Generate Answer Node created!")

except Exception as e:

    print("Node Creation Error:")
    print(e)

Generate Answer Node created!


In [13]:
try:

    state = {
        "question": "What is the purpose of DNS?"
    }

    state = embed_question(state)

    state = retrieve_chunks(state)

    state = generate_answer(state)

    print("\nFinal Answer:\n")
    print(state["answer"])

except Exception as e:

    print("Pipeline Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

The Domain Name System (DNS) serves as the phone book for the Internet by translating human-friendly computer hostnames into IP addresses. It provides a naming system for computers, services, and other resources on the Internet or other Internet Protocol (IP) networks, and associates various information with domain names. The main purpose of DNS is to translate readily memorized domain names to the numerical IP addresses needed for locating and identifying computer services and devices with the underlying network protocols.


In [14]:
try:

    graph_builder = StateGraph(GraphState)

    graph_builder.add_node(
        "embed_question",
        embed_question
    )

    graph_builder.add_node(
        "retrieve_chunks",
        retrieve_chunks
    )

    graph_builder.add_node(
        "generate_answer",
        generate_answer
    )

    print("Nodes added successfully!")

except Exception as e:

    print("Graph Creation Error:")
    print(e)

Nodes added successfully!


In [15]:
try:

    graph_builder.add_edge(
        START,
        "embed_question"
    )

    graph_builder.add_edge(
        "embed_question",
        "retrieve_chunks"
    )

    graph_builder.add_edge(
        "retrieve_chunks",
        "generate_answer"
    )

    graph_builder.add_edge(
        "generate_answer",
        END
    )

    print("Edges connected successfully!")

except Exception as e:

    print("Edge Error:")
    print(e)

Edges connected successfully!


In [16]:
try:

    graph = graph_builder.compile()

    print("LangGraph compiled successfully!")

except Exception as e:

    print("Compilation Error:")
    print(e)

LangGraph compiled successfully!


In [18]:
try:

    result = graph.invoke(
        {
            "question":
            "What is the purpose of DNS?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except Exception as e:

    print("Execution Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

The Domain Name System (DNS) serves as the phone book for the Internet by translating human-friendly computer hostnames into IP addresses. It provides a naming system for computers, services, and other resources on the Internet or other Internet Protocol (IP) networks, and associates various information with domain names. The DNS delegates the responsibility of assigning domain names and mapping those names to Internet resources, and it defines the technical functionality of the database service that is at its core. The main purpose of the DNS is to translate readily memorized domain names to the numerical IP addresses needed for locating and identifying computer services and devices with the underlying network protocols.


In [19]:
result = graph.invoke(
    {
        "question":
        "What is the difference between TCP and UDP?"
    }
)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
The main differences between TCP and UDP are:

1. Connection-oriented vs Connectionless: TCP is a connection-oriented protocol that requires a handshake to establish a connection, while UDP is a connectionless protocol that does not require a handshake.

2. Reliability: TCP is reliable and ensures that data arrives in-order with minimal error, while UDP is unreliable and does not guarantee that data will arrive at its destination.

3. Ordered: TCP ensures that data arrives in-order, while UDP does not guarantee the order of arrival.

4. Heavyweight vs Lightweight: TCP is heavyweight, requiring three packets to set up a connection, while UDP is lightweight and does not require a connection setup.

5. Error control and congestion control: TCP includes error control and congestion control, while UDP does not have built-in error control or congestion control.

6. Use cases: TCP is typically used for applic

In [20]:
result = graph.invoke(
    {
        "question":
        "What is the role of the OSI model?"
    }
)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
The OSI model "provides a common basis for the coordination of standards development for the purpose of systems interconnection." It allows transparent communication through equivalent exchange of protocol data units (PDUs) between two parties, enabling peer-to-peer networking. The model describes communications from the physical implementation to the highest-level representation of data of a distributed application, and it serves as a reference for teaching and documentation.


In [21]:
result = graph.invoke(
    {
        "question":
        "How does HTTP work?"
    }
)

print(result["answer"])

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
HTTP is a request-response protocol in the client-server model. A transaction starts with a client submitting a request to the server, and the server attempts to satisfy the request and returns a response to the client that describes the disposition of the request and optionally contains a requested resource. The client, often a web browser, sends an HTTP request message to the server, and the server sends back an HTTP response message, which includes headers and a body if required. The body of the response message is typically the requested resource, although an error message or other information may also be returned.
